## 6.3 코딩 에이전트 만들기

### 6.3.1 [실습] 사용자 도구 정의하기

In [1]:
# [방법 1] 가장 기본적인 @tool 데코레이터 사용법
# 비유: 평범한 파이썬 함수에 'AI 전용 도구 스티커(@tool)'를 붙이는 가장 간단한 방법입니다.
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# AI가 이 도구를 볼 때 읽게 되는 3가지 핵심 정보(이름, 설명, 입력 인자 구조)를 출력해 봅니다.
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [2]:
# AI 모델에게 실제로 전달되는 전체 JSON 설계도(Schema)를 확인합니다.
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [7]:
# [방법 2] Pydantic(BaseModel)으로 각 입력값(a, b)에 상세 설명(description) 달아주기
# 비유: 신청서 양식(CalculatorInput)을 따로 만들어 'a 칸에는 첫 번째 숫자를 적으세요'라고 친절히 안내하는 방법입니다.
from pydantic import BaseModel, Field
from langchain.tools import tool


class CalculatorInput(BaseModel):
    a: int = Field(description="first number")
    b: int = Field(description="second number")


@tool(args_schema=CalculatorInput)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# args 출력 결과를 보면 a와 b에 'description'이 추가된 것을 확인할 수 있습니다!
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'description': 'first number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'second number', 'title': 'B', 'type': 'integer'}}


In [8]:
# CalculatorInput 설계도가 적용된 JSON 스키마를 확인합니다.
print(multiply.args_schema.model_json_schema())

{'properties': {'a': {'description': 'first number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'second number', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInput', 'type': 'object'}


In [9]:
# [방법 3] parse_docstring=True 옵션으로 함수 설명글(Docstring)에서 인자 설명 자동 추출하기!
# 비유: 별도의 클래스(BaseModel)를 만들지 않고, 함수 안의 Args: 주석만 잘 적어두면 알아서 설명서로 만들어주는 편리한 방식입니다.
@tool(parse_docstring=True)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b


print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'description': 'The first number.', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number.', 'title': 'B', 'type': 'integer'}}


In [10]:
# Docstring 파싱 방식으로 만들어진 최종 JSON 스키마를 확인합니다.
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers.', 'properties': {'a': {'description': 'The first number.', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number.', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [ ]:
# (빈 셀)